# Persistent semantic search

Install `requirements.txt` first and configure a fresh OpenAI key in `.env`. One document represents one book, and its ISBN/category are metadata. Indexing is paid on the first run; subsequent runs reuse the index.


In [ ]:
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv
from search_index import load_or_create_index

ROOT = Path.cwd()
load_dotenv(ROOT / ".env")
books = pd.read_csv(ROOT / "books_with_categories.csv")


In [ ]:
db_books = load_or_create_index(
    books, ROOT / ".cache" / "chroma",
    os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
)


In [ ]:
docs = db_books.similarity_search(
    "A book to teach children about nature", k=10,
    filter={"simple_categories": "Children\'s Nonfiction"},
)
ids = [doc.metadata["isbn13"] for doc in docs]
books.set_index("isbn13").reindex(ids)[["title", "authors", "description"]]
